# Instagram 데이터 중복 정제

같은 `id`가 여러 키워드에 포함된 경우, 지정한 키워드 우선순위가 가장 높은 행 하나만 남깁니다. 원본 파일은 수정하지 않고 `instagram_cleaned.csv`로 저장합니다.

In [5]:
from pathlib import Path
import pandas as pd

# 노트북을 프로젝트 루트 또는 하위 폴더 어디에서 실행해도 CSV를 찾도록 설정
relative_csv_path = Path("02_csv") / "02_instagram" / "instagram_merged.csv"
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / relative_csv_path).exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {relative_csv_path}")

input_path = project_root / relative_csv_path
output_path = input_path.with_name("instagram_cleaned.csv")

df = pd.read_csv(input_path, dtype={"id": "string"})
print(f"전체 행 수: {len(df):,}")
df.head()

전체 행 수: 20,335


,id,caption,media_type,permalink,timestamp,like_count,comments_count,keyword
0,17898709716673599,#먼작귀 #치이카와 #코스프레 #갑옷씨 #포셰트갑옷씨 #포쉐트갑옷씨 #수원 #수원메...,CAROUSEL_ALBUM,https://www.instagram.com/p/De9AmC6mlSY/,2026-10-26 18:55:23+09:00,15.0,1,갑옷씨
1,17878685700651547,고블린과 갑옷씨💟\r\n치킨든 고블린과 쫌 더커진 고블린까지〰️\r\n갑옷아저씨도 ...,CAROUSEL_ALBUM,https://www.instagram.com/p/De_o-1lEwRB/,2026-10-27 19:26:47+09:00,4.0,2,갑옷씨
2,18103745312140350,🌱뭔가 작고 귀엽다\r\n\r\n지치고 팍팍한 일상 속에서 요즘 에디터가 가장 위로...,CAROUSEL_ALBUM,https://www.instagram.com/p/DazEHTlAWLR/,2026-07-15 12:08:10+09:00,18083.0,54,갑옷씨
3,18290126140292291,맥도날드 레스토랑(?)에 온 치이와 포쉐트갑옷씨\r\nちいかわレストランじゃなくて、マ...,IMAGE,https://www.instagram.com/p/De87JZHKcxr/,2026-10-26 18:08:48+09:00,27.0,6,갑옷씨
4,17924891337200386,먼작귀 토벌 전🍀\r\n\r\n홍대 미니소를 갔는데🐢\r\n\r\n역시나 한개도 못...,CAROUSEL_ALBUM,https://www.instagram.com/p/DeV7BvbE1PG/,2026-10-11 14:36:21+09:00,NaN,0,갑옷씨


In [6]:
# id 기준 중복 현황 확인
duplicate_mask = df.duplicated(subset="id", keep=False)
duplicate_rows = df.loc[duplicate_mask].sort_values(["id", "keyword"])

print(f"중복 행 수: {duplicate_mask.sum():,}")
print(f"중복 id 수: {duplicate_rows['id'].nunique(dropna=False):,}")
duplicate_rows[["id", "keyword"]].head(30)

중복 행 수: 11,060
중복 id 수: 4,189


,id,keyword
4559,17842717001944452,모몽가
10676,17842717001944452,우사기
19667,17842717001944452,하치와레
52,17845506996133912,갑옷씨
5504,17845506996133912,모몽가
16466,17845506996133912,쿠리만쥬
10054,17846060220267309,우사기
19504,17846060220267309,하치와레
5569,17846275230645592,모몽가
16599,17846275230645592,쿠리만쥬


In [7]:
# 위에 있을수록 우선순위가 높음
keyword_priority = [
    "치이카와",
    "먼작귀",
    "하치와레",
    "우사기",
    "랏코",
    "모몽가",
    "아노코",
    "데카츠요",
    "시사",
    "갑옷씨",
    "쿠리만쥬",
    "카니",
    "파자마 파티즈",
]

# 공백 유무와 관계없이 같은 키워드로 처리 (예: '파자마 파티즈' == '파자마파티즈')
normalize_keyword = lambda value: "".join(str(value).split())
priority_map = {normalize_keyword(keyword): rank for rank, keyword in enumerate(keyword_priority)}
normalized_keyword = df["keyword"].astype("string").str.replace(r"\s+", "", regex=True)
unknown_keywords = sorted(
    df.loc[normalized_keyword.notna() & ~normalized_keyword.isin(priority_map), "keyword"]
    .astype(str)
    .unique()
)
if unknown_keywords:
    print("우선순위에 없는 keyword(가장 낮은 우선순위로 처리):", unknown_keywords)

work = df.assign(
    _keyword_priority=normalized_keyword.map(priority_map).fillna(len(priority_map)),
    _row_order=range(len(df)),
)

# 우선순위가 높은 행을 먼저 배치해 id별 첫 행만 남긴 뒤, 원래 행 순서로 복원
cleaned_df = (
    work.sort_values(["_keyword_priority", "_row_order"], kind="stable")
    .drop_duplicates(subset="id", keep="first")
    .sort_values("_row_order")
    .drop(columns=["_keyword_priority", "_row_order"])
    .reset_index(drop=True)
)

assert not cleaned_df["id"].duplicated().any(), "id 중복이 남아 있습니다."
cleaned_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"정제 전: {len(df):,}행")
print(f"정제 후: {len(cleaned_df):,}행")
print(f"제거된 중복 행: {len(df) - len(cleaned_df):,}행")
print(f"저장 경로: {output_path}")
cleaned_df.head()

정제 전: 20,335행
정제 후: 13,464행
제거된 중복 행: 6,871행
저장 경로: c:\Users\user\Desktop\KDT_2026\프로젝트\02_데이터_프로젝트\02_csv\02_instagram\instagram_cleaned.csv


,id,caption,media_type,permalink,timestamp,like_count,comments_count,keyword
0,17898709716673599,#먼작귀 #치이카와 #코스프레 #갑옷씨 #포셰트갑옷씨 #포쉐트갑옷씨 #수원 #수원메...,CAROUSEL_ALBUM,https://www.instagram.com/p/De9AmC6mlSY/,2026-10-26 18:55:23+09:00,15.0,1,갑옷씨
1,17878685700651547,고블린과 갑옷씨💟\r\n치킨든 고블린과 쫌 더커진 고블린까지〰️\r\n갑옷아저씨도 ...,CAROUSEL_ALBUM,https://www.instagram.com/p/De_o-1lEwRB/,2026-10-27 19:26:47+09:00,4.0,2,갑옷씨
2,18290126140292291,맥도날드 레스토랑(?)에 온 치이와 포쉐트갑옷씨\r\nちいかわレストランじゃなくて、マ...,IMAGE,https://www.instagram.com/p/De87JZHKcxr/,2026-10-26 18:08:48+09:00,27.0,6,갑옷씨
3,17927782031847428,행복💚\r\n중복없이 한번에 뽑아버리긔~✌🏻\r\n\r\n#먼작귀 #먼가작고귀여운녀...,IMAGE,https://www.instagram.com/p/DBG6SOuvhUi/,2024-10-14 23:30:50+09:00,9.0,1,갑옷씨
4,17919611544449235,치이카와 갑옷씨가 될 수 있다고?!💰✨\r\n치이카와 파크에서 10월 16일부터\r...,CAROUSEL_ALBUM,https://www.instagram.com/p/DeeJo31oxeY/,2026-10-14 19:17:57+09:00,2864.0,17,갑옷씨


In [8]:
cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13464 entries, 0 to 13463
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              13464 non-null  string 
 1   caption         13326 non-null  str    
 2   media_type      13464 non-null  str    
 3   permalink       13464 non-null  str    
 4   timestamp       13464 non-null  str    
 5   like_count      11300 non-null  float64
 6   comments_count  13464 non-null  int64  
 7   keyword         13464 non-null  str    
dtypes: float64(1), int64(1), str(5), string(1)
memory usage: 8.4 MB
